In [ ]:
# Word2Vec : 단어를 희소한 형태 대신, 의미를 반영한 저차원의 밀집벡터로 표현하는 모델
# !pip install gensim
# 영문에서 gensim이 필수

from gensim.models import word2vec    # TfidfModel 중요

sentences = [['king', 'queen', 'man', 'woman'], ['apple', 'banana', 'fruit']]
model = word2vec.Word2Vec(sentences, vector_size=10, window=2, min_count=1, sg=1)
# vector_size: 벡터 차원, window: 문맥윈도우 크기(중심 단어 기준에서 오왼 2개), sg: 모델학습 방법 1(skip-gram) 0(CBOW)
# skip-gram 중심단어로 주변 단어 예측, CBOW 주변 단어를 이용해서 중심단어 예측

print(model.wv['apple'])    # 벡터화 결과 확인
print(model.wv.similarity('king', 'man'))    # 코사인 유사도 계산 결과
print(model.wv.similarity('king', 'apple'))
print(model.wv.similarity('banana', 'apple'))

In [ ]:
sentences2 = [["python", "lan", "program", "computer", "say"]]
model2 = word2vec.Word2Vec(sentences2, vector_size=50, window=2, min_count=1, sg=1, alpha=0.025)    # alpha : learning rate. 
# SGD(확률적 경사하강법)를 이용해 손실 cost를 최소화한다
print(model2.wv)    # KeyedVectors
print(f"인덱스 사전(vocab) : {model2.wv.key_to_index}")
print(f"keys : {model2.wv.key_to_index.keys()}")
print(f"values : {model2.wv.key_to_index.values()}")

print()
vocabs = model2.wv.key_to_index.keys()    # 단어 사전 기억
wordvec_list = [model2.wv[i] for i in vocabs]
# print(wordvec_list)
print(len(wordvec_list[0]))
print(wordvec_list[0])

In [ ]:
# 단어 유사도 확인
print(model2.wv.similarity(w1='python', w2='computer'))
print(model2.wv.most_similar('python', topn=2))

# 시각화
# !pip install koreanize-matplotlib
import matplotlib.pyplot as plt
import koreanize_matplotlib

def plotFunc(vocabs, x,y):
  plt.figure(figsize=(8, 6))
  plt.scatter(x, y)
  for i, v in enumerate(vocabs):
    plt.annotate(v, xy=(x[i], y[i]))

from sklearn.decomposition import PCA
pca = PCA(n_components=2)
xytrans = pca.fit_transform(wordvec_list)
xs = xytrans[:, 0]
ys = xytrans[:, 1]
plotFunc(vocabs, xs, ys)
plt.show()
plt.close()

import numpy as np
print(np.degrees(np.arccos(0.16563551127910614)))    # 80.46584540892889 ; python과 program 사이의 각도가 80.5도

# 유사도 순으로 정렬해서 가까움 정도를 텍스트로 표현
target = 'python'
sim = {w:model2.wv.similarity(target, w) for w in vocabs if w != target}
sort_sim = sorted(sim.items(), key=lambda x:x[1], reverse=True)
print(f"'{target}' 기준 단어별 코사인 유사도\n")
for word, s in sort_sim:
  bar = '⬛️' * int((s + 1) * 10)
  print(f"{word:10} | {bar:20}({s:.3f})")